# 07 · AI chat, six perspectives, and accountable orchestration

**Decision question.** How can an assistant support deliberation while preserving provenance, disagreement, and human decision authority?

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays and tables. Run cells from top to bottom.
The shared `rewiring.py` module must stay beside the notebooks. Initial package downloads require internet;
the core calculations use reproducible synthetic data. No health-service credentials are needed.

**Data contract:** the fictional grid is geographically anchored near Gaborone for teaching map coordinates.
Its people, facilities, climate, events, and model parameters are invented. It is **not a map of actual
Gaborone disease, vulnerability, buildings, or care access**. Satellite catalog metadata, when used, is labeled separately.

[Book](https://jltobias.github.io/JupyterLite-Rewiring-Global-Health/) · [Interactive demo gallery](https://jltobias.github.io/JupyterLite-Rewiring-Global-Health/demos/) · [Data and credits](https://github.com/jltobias/JupyterLite-Rewiring-Global-Health/blob/main/DATA_SOURCES.md)

In [1]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['numpy', 'pandas', 'matplotlib', 'ipywidgets'])
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, IFrame
from rewiring import (city, long_table, map_image, map_matrix, geojson,
                      leaflet, save_export, simulate_twin, teaching_review, HATS, SITE, LABEL, EXTENT)
plt.rcParams.update({'figure.dpi': 105, 'axes.spines.top': False, 'axes.spines.right': False})
cells, cube = city()
panel = long_table(cells, cube)
print(LABEL)
print('Python', sys.version.split()[0], '| cells:', len(cells), '| monthly observations:', len(panel))

SYNTHETIC teaching data · not observations or a forecast
Python 3.13.14 | cells: 64 | monthly observations: 1536


## Six perspectives, with an explicit synthesis step

Edward de Bono's Six Thinking Hats is a human parallel-thinking method. This notebook uses the hats as
computational review roles inspired by your supplied global-health diagram. The purple synthesis step is
a project-specific extension, not an original seventh hat. Template responses below are deterministic;
they do not claim to be language-model inference or authentic community testimony.

![Six perspectives and synthesis](assets/hats-review.svg)

In [2]:
summary={'data_status':'synthetic','cells':len(cells),'mean_heat_c':round(float(cube['heat_c'].mean()),2),
         'population':int(cells.population.sum())}
question='What evidence would justify a neighborhood heat-outreach pilot?'
reviews=teaching_review(question,summary)
display(pd.DataFrame(reviews)[['role','consideration','evidence']])

,role,consideration,evidence
0,White / evidence,"Separate measured data, assumptions, and missi...",Synthetic example only; no validated intervent...
1,Red / experience,Invite community concerns; never fabricate com...,Synthetic example only; no validated intervent...
2,Black / caution,"Inspect disclosure, exclusion, error, and fals...",Synthetic example only; no validated intervent...
3,Yellow / benefit,Name a testable benefit and who should receive...,Synthetic example only; no validated intervent...
4,Green / alternatives,Compare non-AI options and less invasive ways ...,Synthetic example only; no validated intervent...
5,Blue / process,"Assign human review, evaluation, and a stoppin...",Synthetic example only; no validated intervent...


## A chat interface that makes its mode visible

The widget accepts a question and produces structured teaching prompts. It sends nothing to an external
model. Do not enter sensitive personal information. An ordinary Python call above gives the same result
when widgets are unavailable in a static book or assistive workflow.

In [3]:
import ipywidgets as widgets
entry=widgets.Textarea(value=question,description='Question:',layout=widgets.Layout(width='95%',height='80px'))
button=widgets.Button(description='Review (template)',button_style='info')
output=widgets.Output()
def review_click(_):
    with output:
        output.clear_output(wait=True)
        display(pd.DataFrame(teaching_review(entry.value[:1000],summary))[['role','consideration']])
button.on_click(review_click)
display(widgets.VBox([widgets.HTML('<b>Teaching mode · deterministic · no model call</b>'),entry,button,output]))

## Preserve disagreement instead of averaging it away

Orchestration means coordinating roles and evidence, not turning a majority vote into truth. A privacy
objection can require changing the analysis even if the opportunity is attractive. A model should not
infer what a community feels: that perspective requires engagement with actual people.

In [4]:
record={'question':question,'data_summary':summary,
 'agreement':['Use aggregate evidence','Evaluate outreach reach and unintended effects'],
 'unresolved':['Which communities are absent from the data?','Can services meet newly identified demand?'],
 'next_actions':['Seek community priorities','Check denominators','Test a bounded pilot'],
 'human_owner':'Workshop team; replace with a named accountable owner in actual work',
 'decision_status':'Not approved for operational use'}
display(pd.Series(record))
print(save_export('deliberation.json',record))

question           What evidence would justify a neighborhood hea...
data_summary       {'data_status': 'synthetic', 'cells': 64, 'mea...
agreement          [Use aggregate evidence, Evaluate outreach rea...
unresolved         [Which communities are absent from the data?, ...
next_actions       [Seek community priorities, Check denominators...
human_owner        Workshop team; replace with a named accountabl...
decision_status                     Not approved for operational use
dtype: object

exports\deliberation.json


## JupyterLite AI is included in the site

The build installs **jupyterlite-ai**, the browser-capable chat/completion extension, which is distinct
from server-dependent integrations. Open its chat panel in JupyterLite. No provider credentials or model
endpoint are shipped in this public site. Provider setup is required before real model inference.

For institutional use, configure an authenticated, approved model gateway and keep long-lived provider
secrets on the server. Review notebook-context sharing, retention, authorization, and costs before enabling
a model. The extension's secrets manager defaults to in-memory key handling; disabling it can persist
keys in browser settings. An in-memory browser key is still accessible to the client.

[Extension documentation](https://jupyterlite-ai.readthedocs.io/en/latest/) ·
[API-key handling](https://jupyterlite-ai.readthedocs.io/en/latest/api-keys/) ·
[custom provider integration](https://jupyterlite-ai.readthedocs.io/en/latest/custom-providers/).

## An explicit gateway request contract

This adapter is for a gateway **you implement and govern**, not a claim that GitHub Pages provides an AI API.
The request includes only the known synthetic summary and question. A schema is not a privacy detector:
free text can still contain sensitive material. Do not enable it until the endpoint and authorization are reviewed.
No model response is executed as code, SQL, or a tool command.

In [5]:
async def ask_gateway(endpoint, question, summary):
    from urllib.parse import urlparse
    if urlparse(endpoint).scheme != 'https': raise ValueError('Use an approved HTTPS endpoint')
    teaching_review(question,summary)  # explicit aggregate input contract, not content screening
    payload={'schema_version':1,'question':question,'summary':summary,
             'response_contract':{'answer':'string','sources':'list of strings','uncertainties':'list of strings'}}
    if sys.platform != 'emscripten':
        raise RuntimeError('This sample adapter uses browser authenticated fetch; implement a native client separately')
    from pyodide.http import pyfetch
    response=await pyfetch(endpoint,method='POST',headers={'Content-Type':'application/json'},
                           credentials='include',body=json.dumps(payload))
    if response.status!=200: raise RuntimeError(f'Gateway returned HTTP {response.status}')
    result=await response.json()
    if (not isinstance(result,dict) or not isinstance(result.get('answer'),str)
        or not all(isinstance(result.get(k),list) and all(isinstance(v,str) for v in result[k])
                   for k in ['sources','uncertainties'])):
        raise ValueError('Gateway response violates the contract')
    return result
# No endpoint is configured and no request runs by default.
print('Gateway adapter defined; no model calls made.')

Gateway adapter defined; no model calls made.


## Investigate, interpret, and discuss

1. Add a dissent field to the synthesis record and preserve a minority concern.
2. Try a request that should remain a human decision; write the boundaries of acceptable assistance.
3. Review an assistant answer for invented citations, causal overclaims, and requests for unnecessary precision.
4. Explain the difference between schema validation, authentication, privacy review, and factual verification.

**Before sharing:** state the decision, data status, units, denominator, scale, uncertainty, and who can contest the result.
Record what would change your conclusion. Export only information that the intended audience needs.

## Sources and attribution

- de Bono, E. (1985), *Six Thinking Hats*.
- [Related Six Hats agents project](https://github.com/jltobias/JupyterLite-AI-Agents-and-Orchestrator-Six-Thinking-Hats).
- [Project Jupyter: AI in Jupyter](https://jupyter.org/ai) and [jupyterlite-ai](https://github.com/jupyterlite/ai).
- Supplied global-health Six Thinking Hats image: conceptual inspiration; this lesson's SVG is an original explanatory drawing.

Original lesson code: MIT. Original narrative, synthetic data, and generated figures: CC BY 4.0.
Third-party data, videos, services, and software retain their own terms.
[Full references](https://github.com/jltobias/JupyterLite-Rewiring-Global-Health/blob/main/REFERENCES.md) · [Third-party notices](https://github.com/jltobias/JupyterLite-Rewiring-Global-Health/blob/main/THIRD_PARTY_NOTICES.md).